# XGBoost (Extreme Gradient Boosting) Classifier & Regressor

**XGBoost**, short for **Extreme Gradient Boosting**, is an optimized distributed gradient boosting library designed to be highly efficient, flexible, and portable. It implements machine learning algorithms under the Gradient Boosting framework, but includes several critical mathematical improvements (like second-order Taylor expansion and leaf weight regularization) and system optimizations (like parallel tree building and cache-aware access).

In these detailed notes, we will cover:
1. **Core Concept:** Why XGBoost is "Extreme"—including regularization, Taylor approximation, and sparsity handling.
2. **Mathematical Formulation (The Regularized Objective):** Step-by-step derivation of the regularized objective function, gradients, hessians, and optimal leaf weights.
3. **Split Finding & Pruning (Gain):** How nodes are split and pruned using the Gain formula and leaf weight penalty ($\gamma$).
4. **XGBoost for Regression vs. Classification:** Gradients, hessians, loss functions, and prediction space.
5. **Key Hyperparameters:** Shrinkage (`eta`), `max_depth`, `min_child_weight`, `gamma`, `lambda`, `alpha`, and subsampling.
6. **Practical Python Demos:**
   - **Demo A:** Step-by-Step XGBoost Regressor implementation from scratch (custom node split and leaf weight calculation) on a toy dataset.
   - **Demo B:** Visualizing the effect of $L2$ regularization ($\lambda$) and pruning ($\gamma$) on decision boundaries.
   - **Demo C:** Training & tuning `XGBClassifier` and `XGBRegressor` using `GridSearchCV` from the `xgboost` package.
   - **Summary Checklist** for XGBoost.

## 1. Core Concepts & Innovations

Standard Gradient Boosting builds trees sequentially by fitting residuals. XGBoost takes this framework and makes it robust and extremely fast using several innovations:

### Regularized Objective Function
Unlike standard GBMs, XGBoost adds $L1$ ($\alpha$) and $L2$ ($\lambda$) regularization directly to the leaf weights in the objective function to control model complexity and prevent overfitting.

### Second-Order Taylor Expansion
Standard GBMs require custom mathematical derivations for each loss function to compute pseudo-residuals. XGBoost uses a **second-order Taylor series approximation** of the loss function, allowing it to optimize *any* custom differentiable loss function using only its first (gradient) and second (hessian) derivatives.

### Weighted Quantile Sketch
Instead of sorting all feature values to find the best split (which is computationally expensive on large datasets), XGBoost uses a weighted quantile sketch algorithm to propose candidate split points based on the distribution of feature values.

### Sparsity-Aware Split Finding
Real-world datasets often have missing values (sparse matrices). XGBoost automatically learns a default direction (left or right) for missing values at each split node based on which direction reduces training loss the most.

### System Optimizations
- **Column Block Parallelism:** Feature values are stored in sorted blocks in memory, enabling parallel calculation of split gains.
- **Cache-Aware Access:** Buffers gradients and hessians in cache to speed up computations.
- **Out-of-Core Computing:** Optimizes disk read/write throughput for datasets too large to fit in RAM.

## 2. Mathematical Formulation: The Regularized Objective

At iteration $t$, the prediction for sample $i$ is $\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + f_t(x_i)$, where $f_t$ is the new tree we want to add.
The objective function to minimize at step $t$ is:

$$\mathcal{L}^{(t)} = \sum_{i=1}^{n} L\left(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)\right) + \Omega(f_t)$$

Where the regularization term $\Omega(f_t)$ penalizes the tree structure:

$$\Omega(f_t) = \gamma T + \frac{1}{2} \lambda \sum_{j=1}^{T} w_j^2 + \alpha \sum_{j=1}^{T} |w_j|$$

- $T$: Number of leaf nodes.
- $w_j$: Weight (output value) of leaf $j$.
- $f_t$: The functions constructed by tree models.
- $\gamma$: Complexity penalty for adding a new leaf (acts as a threshold for pruning).
- $\lambda, \alpha$: $L2$ and $L1$ regularization coefficients on leaf weights.

---

### Second-Order Taylor Series Approximation
To optimize the objective for any loss function $L$, we approximate the loss using a Taylor series up to the second order around the previous prediction $\hat{y}_i^{(t-1)}$:

$$L(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)) \approx L(y_i, \hat{y}_i^{(t-1)}) + g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i)$$

Where:
- $g_i = \left[ \frac{\partial L(y_i, \hat{y}_i)}{\partial \hat{y}_i} \right]_{\hat{y}_i = \hat{y}_i^{(t-1)}}$ (First derivative / Gradient)
- $h_i = \left[ \frac{\partial^2 L(y_i, \hat{y}_i)}{\partial \hat{y}_i^2} \right]_{\hat{y}_i = \hat{y}_i^{(t-1)}}$ (Second derivative / Hessian)

Removing the constant term $L(y_i, \hat{y}_i^{(t-1)})$ (which doesn't depend on the new tree $f_t$), the simplified objective at step $t$ becomes:

$$\tilde{\mathcal{L}}^{(t)} = \sum_{i=1}^n \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \gamma T + \frac{1}{2} \lambda \sum_{j=1}^T w_j^2$$

---

### Aggregating by Leaves
We can rewrite the sum over individual training instances $i$ as a sum over leaf nodes $j$. Let $I_j$ be the set of indices of training samples that fall into leaf node $j$. The function $f_t(x_i)$ outputs the weight $w_j$ for any sample $i \in I_j$:

$$\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^{T} \left[ \left( \sum_{i \in I_j} g_i \right) w_j + \frac{1}{2} \left( \sum_{i \in I_j} h_i + \lambda \right) w_j^2 \right] + \gamma T$$

Let's define:
- $G_j = \sum_{i \in I_j} g_i$ (Sum of gradients in leaf $j$)
- $H_j = \sum_{i \in I_j} h_i$ (Sum of hessians in leaf $j$)

The objective simplifies to:

$$\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^{T} \left[ G_j w_j + \frac{1}{2} (H_j + \lambda) w_j^2 \right] + \gamma T$$

---

### Solving for Optimal Leaf Weights
To find the optimal weight $w_j^*$ for leaf $j$, we take the partial derivative of $\tilde{\mathcal{L}}^{(t)}$ with respect to $w_j$, set it to zero, and solve:

$$\frac{\partial}{\partial w_j} \left( G_j w_j + \frac{1}{2} (H_j + \lambda) w_j^2 \right) = G_j + (H_j + \lambda) w_j = 0$$

$$w_j^* = -\frac{G_j}{H_j + \lambda}$$

---

### Structure Score (Quality of Tree Structure)
Substituting $w_j^*$ back into the objective function yields the optimal objective value (also called the **structure score** or **quality score** of the tree):

$$\tilde{\mathcal{L}}^{(t)*} = -\frac{1}{2} \sum_{j=1}^{T} \frac{G_j^2}{H_j + \lambda} + \gamma T$$

A smaller (more negative) score indicates a better tree structure.

## 3. Split Finding & Pruning: The Gain Formula

When building a tree, we need to decide how to split a parent node into left ($L$) and right ($R$) child nodes. XGBoost evaluates candidate splits by measuring the reduction in the objective score (increase in quality). The **Gain** of a split is defined as:

$$\text{Gain} = \text{Score}_{Left} + \text{Score}_{Right} - \text{Score}_{Parent}$$

Using our structure score formula (excluding $\gamma$ for parent, and adding $\gamma$ penalties for child leaves):

$$\text{Gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{G_P^2}{H_P + \lambda} \right] - \gamma$$

Where:
- $G_P = G_L + G_R$ and $H_P = H_L + H_R$ are the sums of gradients and hessians in the parent node.
- $\gamma$ is the penalty for introducing a new split (which increases the number of leaves by 1).

### Post-Pruning
If the maximum Gain for a node split is negative (i.e., the best split yields $\text{Gain} < 0$), it means the loss reduction does not cover the complexity penalty $\gamma$.
XGBoost handles pruning as follows:
1. Build the tree to its maximum depth (`max_depth`).
2. Prune the tree bottom-up. For any node, if the split results in a negative Gain, we collapse the child nodes back into a single leaf.

## 4. XGBoost for Regression vs. Classification

Because XGBoost uses Taylor expansion, we only need to specify the loss function, and compute its gradient $g_i$ and hessian $h_i$ at each step:

| Task / Loss | Loss Function $L(y_i, \hat{y}_i)$ | Prediction Space $\hat{y}_i$ | Gradient $g_i$ | Hessian $h_i$ | Optimal Leaf Weight $w_j^*$ |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Regression** (MSE) | $\frac{1}{2}(y_i - \hat{y}_i)^2$ | Continuous Value | $\hat{y}_i - y_i$ | $1$ | $\frac{\sum_{i \in I_j} (y_i - \hat{y}_i)}{|I_j| + \lambda}$ |
| **Classification** (Log Loss) | $-y_i \ln(p_i) - (1-y_i) \ln(1-p_i)$ where $p_i = \sigma(\hat{y}_i)$ | Log-odds (continuous) | $p_i - y_i$ | $p_i(1 - p_i)$ | $\frac{\sum_{i \in I_j} (y_i - p_i)}{\sum_{i \in I_j} p_i(1 - p_i) + \lambda}$ |

## 5. Key Hyperparameters

*   **`learning_rate` / `eta`:** Step size shrinkage used to prevent overfitting. Scales tree contributions: $F_t = F_{t-1} + \eta \cdot f_t(x)$. Typical values: $0.01$–$0.3$.
*   **`max_depth`:** Maximum depth of a tree. Typical values: $3$–$10$.
*   **`min_child_weight`:** Minimum sum of instance Hessian ($H_j$) needed in a child leaf. For classification, $H_j = \sum p_i(1-p_i)$. If the sum of hessians in a leaf is less than `min_child_weight`, split finding stops. Highly effective for regularizing small classes.
*   **`gamma` / `min_split_loss`:** Minimum loss reduction required to make a split. Acts as pruning threshold.
*   **`reg_lambda`:** $L2$ regularization term on leaf weights ($\lambda$). Increasing this forces leaf weights to be closer to 0.
*   **`reg_alpha`:** $L1$ regularization term on leaf weights ($\alpha$). Can lead to sparse weights.
*   **`subsample`:** Fraction of samples to randomly sample for training each tree (Stochastic Gradient Boosting).
*   **`colsample_bytree`:** Fraction of columns (features) to randomly sample for building each tree.

## 6. Code Setup & Imports

Let's prepare the Python environment and import the required libraries. Make sure the `xgboost` library is installed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report

# Try to import xgboost; print notice if missing
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("xgboost package is not installed. Please install it using: pip install xgboost")

# Set visualization contexts
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 7. Demo A: XGBoost Regressor From Scratch

We will build a custom decision tree node (`XGBNode`) and regressor (`SimpleXGBRegressor`) from scratch in Python. The splitting algorithm will search features and thresholds to find the split maximizing the XGBoost Gain formula.

In [ ]:
class XGBNode:
    def __init__(self, reg_lambda=1.0, gamma=0.0, max_depth=3):
        self.reg_lambda = reg_lambda
        self.gamma = gamma
        self.max_depth = max_depth
        self.feature_idx = None
        self.threshold = None
        self.left = None
        self.right = None
        self.weight = None

    def fit(self, X, g, h, depth=0):
        # Calculate optimal leaf weight in case we stop splitting
        G = np.sum(g)
        H = np.sum(h)
        self.weight = -G / (H + self.reg_lambda)

        # Stop splitting if max depth reached or too few samples
        if depth >= self.max_depth or len(X) <= 1:
            return

        best_gain = 0.0
        best_feature = None
        best_threshold = None
        
        n_samples, n_features = X.shape
        G_P = G
        H_P = H
        score_P = (G_P ** 2) / (H_P + self.reg_lambda)
        
        for feat in range(n_features):
            x_col = X[:, feat]
            # Consider unique values in column as threshold candidates
            thresholds = np.unique(x_col)
            for thresh in thresholds:
                left_mask = x_col <= thresh
                right_mask = ~left_mask
                
                # Ensure split contains samples on both sides
                if not np.any(left_mask) or not np.any(right_mask):
                    continue
                
                G_L = np.sum(g[left_mask])
                H_L = np.sum(h[left_mask])
                G_R = np.sum(g[right_mask])
                H_R = np.sum(h[right_mask])
                
                # Compute structure score improvement
                score_L = (G_L ** 2) / (H_L + self.reg_lambda)
                score_R = (G_R ** 2) / (H_R + self.reg_lambda)
                
                # Gain formula
                gain = 0.5 * (score_L + score_R - score_P) - self.gamma
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feat
                    best_threshold = thresh

        # If a split is found with positive Gain, recursively partition
        if best_feature is not None:
            self.feature_idx = best_feature
            self.threshold = best_threshold
            
            x_col = X[:, self.feature_idx]
            left_mask = x_col <= self.threshold
            right_mask = ~left_mask
            
            self.left = XGBNode(reg_lambda=self.reg_lambda, gamma=self.gamma, max_depth=self.max_depth)
            self.right = XGBNode(reg_lambda=self.reg_lambda, gamma=self.gamma, max_depth=self.max_depth)
            
            self.left.fit(X[left_mask], g[left_mask], h[left_mask], depth + 1)
            self.right.fit(X[right_mask], g[right_mask], h[right_mask], depth + 1)

    def predict_row(self, x):
        if self.feature_idx is None:
            return self.weight
        if x[self.feature_idx] <= self.threshold:
            return self.left.predict_row(x)
        else:
            return self.right.predict_row(x)

    def predict(self, X):
        return np.array([self.predict_row(x) for x in X])


class SimpleXGBRegressor:
    def __init__(self, n_estimators=10, learning_rate=0.1, reg_lambda=1.0, gamma=0.0, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.reg_lambda = reg_lambda
        self.gamma = gamma
        self.max_depth = max_depth
        self.base_pred = 0.0 # Initialize predictions at 0
        self.trees = []

    def fit(self, X, y):
        f_m = np.full_like(y, self.base_pred, dtype=np.float64)
        
        for m in range(self.n_estimators):
            # Compute Gradient and Hessian for MSE loss: L = 0.5*(f - y)^2
            g = f_m - y
            h = np.ones_like(y, dtype=np.float64)
            
            tree = XGBNode(reg_lambda=self.reg_lambda, gamma=self.gamma, max_depth=self.max_depth)
            tree.fit(X, g, h)
            
            # Scale and add tree output to predictions
            f_m += self.learning_rate * tree.predict(X)
            self.trees.append(tree)

    def predict(self, X):
        preds = np.full(X.shape[0], self.base_pred, dtype=np.float64)
        for tree in self.trees:
            preds += self.learning_rate * tree.predict(X)
        return preds

# Create non-linear regression data
np.random.seed(42)
X_toy = np.sort(5 * np.random.rand(100, 1), axis=0)
y_toy = np.cos(X_toy).ravel() + np.random.normal(0, 0.1, X_toy.shape[0])

# Fit our scratch model
scratch_xgb = SimpleXGBRegressor(n_estimators=50, learning_rate=0.1, reg_lambda=1.0, gamma=0.01, max_depth=3)
scratch_xgb.fit(X_toy, y_toy)
preds_scratch = scratch_xgb.predict(X_toy)

print(f"Scratch XGBoost MSE: {mean_squared_error(y_toy, preds_scratch):.6f}")

# Visualize the fit
plt.figure(figsize=(10, 6))
plt.scatter(X_toy, y_toy, color='#3498db', alpha=0.7, label='Data')
plt.plot(X_toy, preds_scratch, color='#e74c3c', lw=3, label='Scratch XGBRegressor Fit')
plt.title("XGBoost Regressor from Scratch: Non-linear Fitting", fontsize=13, fontweight='bold')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.show()

## 8. Demo B: Visualizing Regularization ($\lambda$) and Pruning ($\gamma$) Effects

Regularization is the defining difference between XGBoost and standard boosting.
*   **High $L2$ Regularization ($\lambda$):** Suppresses leaf outputs ($w_j^* = -\frac{G_j}{H_j + \lambda}$). Larger values shrink predictions and smooth the boundary.
*   **High Pruning Penalty ($\gamma$):** Requires splits to achieve high Gain to survive. Larger values lead to shallower trees with fewer splits.

Let's visualize these effects.

In [ ]:
# Create grid predictions to visualize boundary changes
X_plot = np.linspace(0, 5, 500).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Case 1: Standard model (low regularization)
model_std = SimpleXGBRegressor(n_estimators=50, learning_rate=0.1, reg_lambda=0.1, gamma=0.0, max_depth=3)
model_std.fit(X_toy, y_toy)
axes[0].scatter(X_toy, y_toy, color='#3498db', alpha=0.5)
axes[0].plot(X_plot, model_std.predict(X_plot), color='red', lw=2.5)
axes[0].set_title("Standard model\n(lambda=0.1, gamma=0.0)", fontweight='bold')

# Case 2: High L2 Regularization (lambda=50)
model_reg = SimpleXGBRegressor(n_estimators=50, learning_rate=0.1, reg_lambda=50.0, gamma=0.0, max_depth=3)
model_reg.fit(X_toy, y_toy)
axes[1].scatter(X_toy, y_toy, color='#3498db', alpha=0.5)
axes[1].plot(X_plot, model_reg.predict(X_plot), color='green', lw=2.5)
axes[1].set_title("High L2 Regularization\n(lambda=50.0, gamma=0.0)", fontweight='bold')

# Case 3: High Pruning Penalty (gamma=0.5)
model_prune = SimpleXGBRegressor(n_estimators=50, learning_rate=0.1, reg_lambda=0.1, gamma=0.5, max_depth=3)
model_prune.fit(X_toy, y_toy)
axes[2].scatter(X_toy, y_toy, color='#3498db', alpha=0.5)
axes[2].plot(X_plot, model_prune.predict(X_plot), color='purple', lw=2.5)
axes[2].set_title("High Pruning Penalty\n(lambda=0.1, gamma=0.5)", fontweight='bold')

plt.suptitle("XGBoost Regularization and Pruning Visualized", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Demo C: Classification & Regression using `xgboost` package

Now we will use the official `xgboost` Python library. We will demonstrate hyperparameter tuning using cross-validation (`GridSearchCV`) for classification and regression tasks.

In [ ]:
if not XGB_AVAILABLE:
    print("Skipping Demo C: xgboost package is not installed.")
else:
    # --- PART 1: XGBoost Classification ---
    # Create classification dataset
    X_clf, y_clf = make_classification(n_samples=500, n_features=8, n_informative=6, random_state=42)
    X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)
    
    # Grid search parameters
    param_grid_c = {
        'n_estimators': [50, 100],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 4],
        'reg_lambda': [1.0, 10.0]
    }
    
    # Train classifier
    print("Running GridSearchCV for XGBClassifier...")
    clf_search = GridSearchCV(
        estimator=xgb.XGBClassifier(eval_metric='logloss', random_state=42),
        param_grid=param_grid_c,
        cv=3,
        scoring='accuracy',
        n_jobs=-1
)
    clf_search.fit(X_train_c, y_train_c)
    
    best_clf = clf_search.best_estimator_
    acc = accuracy_score(y_test_c, best_clf.predict(X_test_c))
    
    print("\nXGBClassifier Results:")
    print("="*60)
    print(f"Best Parameters: {clf_search.best_params_}")
    print(f"Test Accuracy:   {acc:.4f}")
    print("="*60)
    print("\nClassification Report:")
    print(classification_report(y_test_c, best_clf.predict(X_test_c)))
    
    # --- PART 2: XGBoost Regression ---
    # Create regression dataset
    X_reg, y_reg = make_regression(n_samples=500, n_features=6, noise=15.0, random_state=42)
    X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)
    
    param_grid_r = {
        'n_estimators': [50, 100],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 4],
        'reg_lambda': [1.0, 10.0]
    }
    
    # Train regressor
    print("\nRunning GridSearchCV for XGBRegressor...")
    reg_search = GridSearchCV(
        estimator=xgb.XGBRegressor(random_state=42),
        param_grid=param_grid_r,
        cv=3,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )
    reg_search.fit(X_train_r, y_train_r)
    
    best_reg = reg_search.best_estimator_
    mse = mean_squared_error(y_test_r, best_reg.predict(X_test_r))
    
    print("\nXGBRegressor Results:")
    print("="*60)
    print(f"Best Parameters: {reg_search.best_params_}")
    print(f"Test MSE:        {mse:.4f}")
    print("="*60)
    
    # Plot feature importances
    plt.figure(figsize=(10, 5))
    importances = best_clf.feature_importances_
    indices = np.argsort(importances)[::-1]
    sns.barplot(x=importances[indices], y=[f"Col {i}" for i in indices], palette='viridis')
    plt.title("XGBClassifier Feature Importances", fontsize=13, fontweight='bold')
    plt.xlabel('Importance')
    plt.ylabel('Features')
    plt.tight_layout()
    plt.show()

## Summary Checklist for XGBoost

1.  **Objective Function:** Includes loss function and tree complexity penalty:
    $$\mathcal{L} = \sum L(y_i, \hat{y}_i) + \gamma T + \frac{1}{2} \lambda \sum w_j^2$$
2.  **Taylor Expansion:** Uses first-order gradient ($g_i$) and second-order hessian ($h_i$) to approximate loss, supporting any custom differentiable loss.
3.  **Optimal Leaf Weight:**
    $$w_j^* = -\frac{\sum g_i}{\sum h_i + \lambda}$$
4.  **Split Gain:**
    $$\text{Gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{G_P^2}{H_P + \lambda} ight] - \gamma$$
    If $\text{Gain} < 0$, the split is collapsed (post-pruning).
5.  **Classification vs. Regression:**
    -   *Regression (MSE):* $g_i = \hat{y}_i - y_i$, $h_i = 1$.
    -   *Classification (Log Loss):* $g_i = p_i - y_i$, $h_i = p_i(1 - p_i)$.
6.  **Regularization Parameters:**
    -   `reg_lambda` ($\lambda$): Shrinks leaf weights.
    -   `gamma` ($\gamma$): Dictates minimum gain needed to keep a node split.
    -   `min_child_weight`: Dictates minimum sum of hessian ($H_j$) in a leaf.
7.  **Sparsity & Hardware:** Automatically handles missing values and uses out-of-core sorting blocks for hardware speedups.